# Fine-tune Mini-Omni2 on Colab (free T4) (notebook v9)

**Notebook version v9.** Changes:
* **v9** (2026-11-09): round-7 data (86 topics, interview answers, "my responses are limited" fallbacks). A100: batch 8 at lr 1e-5 (batch 32 undertrained the voice). Early stopping with patience 8 on 5% held-out questions; keeps the best weights in `ckpt_v7` and the last ones in `ckpt_v7/last`. Up to 6 epochs.
* v8 (2026-11-06): round-6 data (first person, 68 topics, cloned voice): expects 68 topics.
* v7 (2026-11-06): a settings cell (batch, accumulation, learning rate) filled in from the GPU; A100 default batch 32 at lr 2e-5. Change them there before Run all.
* v6 (2026-11-06): picks precision and batch from the GPU (T4: fp16, batch 2 x 4; L4: bf16, 4 x 2; A100: bf16, 8 x 1). Same effective batch of 8.
* v5 (2026-11-06): RAG data (2 retrieved answers in front of every question); checks the zip has them.
* v4 (2026-11-06): round-4 data, 40 merged topics (expects a zip with 40). Local folder `/content/omni-cv`.
* v3 (2026-11-05): `--typed --cv-weight 1 --epochs 2`, progress on Colab's disk instead of Drive.
* v2: fp16 memory setting. v1: first version.

Runs only step 4 (`04_train.py`). Prepare the data on your own machine first (steps 1 to 3 in the README), then
pack it with `finetune/make_colab_bundle.sh` and upload `omni_cv_train.zip` to a Google Drive folder called `omni-cv`.

1. **Runtime > Change runtime type > T4 GPU**.
2. Run the cells in order. If the browser disconnects, reconnect and Run all: it resumes while the same Colab machine is alive.
3. Expect a few hours on a T4. When it finishes, download the `omni-cv/ckpt_v7` folder from Drive (best weights, plus `last/`: about 6 GB).

In [ ]:
NOTEBOOK_VERSION = 'v9'
EXPECTED_TOPICS = 86  # round 7: interview topics
print('notebook', NOTEBOOK_VERSION)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/omni-cv'

In [ ]:
!rm -rf /content/omni-cv && mkdir -p /content/omni-cv && cd /content/omni-cv && unzip -q $D/omni_cv_train.zip && ls -R /content/omni-cv | head -20

In [ ]:
import torch
ev = torch.load('/content/omni-cv/finetune/work/encoded/eval.pt', weights_only=False)
topics = {r['id'] for r in ev if r['kind'] == 'cv'}
print(f'data: {len(ev)} held-out examples, {len(topics)} CV topics')
assert len(topics) == EXPECTED_TOPICS, f'wrong zip: {len(topics)} topics, expected {EXPECTED_TOPICS}'
assert all('ctx' in r for r in ev), 'wrong zip: no retrieved context (RAG) in it'
print('RAG context: yes')

In [ ]:
from huggingface_hub import hf_hub_download
for f in ['lit_model.pth', 'model_config.yaml', 'tokenizer.json', 'tokenizer_config.json']:
    hf_hub_download('gpt-omni/mini-omni2', f, local_dir='/content/mini-omni2')

Training. Progress is saved every 200 steps to Colab's own disk (`/content`), not to Drive: on a mounted Drive,
every replaced file goes to the Trash and still counts against your storage (6.4 GB per save). If the browser
disconnects, reconnect and Run all: it resumes while the same Colab machine is alive. Only the result goes to
Drive (~3 GB).

In [ ]:
# ---- training settings: filled in from the GPU, change them here if you like ----
import torch
gpu = torch.cuda.get_device_properties(0)
mem = gpu.total_memory / 2**30
AMP = 'bf16' if torch.cuda.get_device_capability(0)[0] >= 8 else 'fp16'  # A100/L4: bf16; T4: fp16
if mem >= 35:   # A100 (batch 32 undertrained the voice in round 6: keep 8)
    BATCH, ACCUM, LR = 8, 1, 1e-5
elif mem >= 20: # L4
    BATCH, ACCUM, LR = 4, 2, 1e-5
else:           # T4
    BATCH, ACCUM, LR = 2, 4, 1e-5
EPOCHS = 6      # an upper limit: early stopping usually ends it sooner
PATIENCE = 8    # validation checks (every 200 steps) without improvement before stopping
print(f'{gpu.name}, {mem:.0f} GB -> --amp {AMP} --batch {BATCH} --accum {ACCUM} --lr {LR} --epochs {EPOCHS} --patience {PATIENCE}')
%cd /content/omni-cv/finetune
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MINI_OMNI2_DIR=/content/mini-omni2 FT_WORK=/content/omni-cv/finetune/work python 04_train.py \
    --amp $AMP --batch $BATCH --accum $ACCUM --epochs $EPOCHS --lr $LR --cv-weight 1 --typed \
    --val-frac 0.05 --val-every 200 --patience $PATIENCE --save-last \
    --state /content/state.pt --save-every 200 --out $D/ckpt_v7

In [ ]:
!ls -la $D/ckpt_v7 $D/ckpt_v7/last